# Etapa 2 — Treinamento do modelo de contagem de gado (qualquer ângulo)

**Objetivo:** treinar um detector YOLO de bovinos com três fontes públicas e medir **quão bem ele conta** em imagens que não viu, **por tipo de cenário**.

**Pré-requisitos:** `python 01_obter_dados.py` (Zenodo, pasto aéreo), `python 03_obter_dados_extras.py` (Open Images e COLO) e, opcionalmente, `python 04_obter_cattleeyeview.py` (CattleEyeView, vista de cima em rampa). Com `USAR_EXTRAS = False` o notebook treina só com o Zenodo.

**Como usar:** rode as células em ordem, uma por vez. Cada seção termina com uma verificação (`assert`) para o erro aparecer onde nasce.

**Escolhas do método (e por quê):**
- **Zenodo (drone):** imagens de 4000×3000 px ou mais, vaca pequena. Treino e inferência em **tiles de 1024 px sem redimensionar**; a contagem soma as detecções dos tiles depois de remover duplicatas nas bordas.
- **COLO e Open Images:** imagens de até 1.152 px de lado e vaca grande. Entram **inteiras**, sem tiles (um tile cortaria o animal); o YOLO ajusta a imagem a 1024 px.
- **Divisão sem vazamento:** por voo (Zenodo); por cena ou sequência (COLO — a divisão train/test dos autores **não** é usada, porque ela vaza; veja a seção 6b); pelos vídeos (CattleEyeView, divisão dos autores); e a divisão oficial (Open Images).
- **Cenário:** cada fonte/ângulo é um cenário (`dominio`). O erro de contagem é medido **por cenário**, em imagens inteiras do conjunto de teste, e o limiar de confiança é escolhido na validação, por cenário. O mAP por tile é secundário.
- **Limite conhecido do COLO:** há vacas visíveis sem caixa (conferido em imagem). Nesse cenário o modelo pode parecer contar a mais do que o "real".
- **Sem dado de Nelore ou pasto brasileiro:** nenhuma fonte aberta com caixas foi encontrada; o desempenho nesse gado não está medido.


## 1. Configuração

In [ ]:
import json, os, random, re, shutil, warnings, zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

import contagem as ct   # contagem.py, na pasta do projeto
from importlib import reload
ct = reload(ct)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_columns", 30)

PROJ = Path.cwd()
RAW, INTERIM, YOLO_DIR, OUT = PROJ / "data" / "raw", PROJ / "data" / "interim", PROJ / "data" / "yolo", PROJ / "data" / "processed"
for p in (INTERIM, YOLO_DIR, OUT):
    p.mkdir(parents=True, exist_ok=True)
ZIP_COMPLETO, ZIP_SELECAO = RAW / "Cattle_drone_images_042024.zip", RAW / "Cattle_drone_images_selecao.zip"
ZIP = ZIP_COMPLETO if ZIP_COMPLETO.exists() else ZIP_SELECAO   # o zip completo, se existir; senão o do modo --seletivo
EXTRAS = PROJ / "data" / "extras"   # saída de 03_obter_dados_extras.py (Open Images + COLO, já em YOLO)

# ---- parâmetros (mude aqui) ----
SEED = 42
TILE = 1024            # tamanho do tile em pixels (nativo, sem redimensionar)
OVERLAP = 0.2          # sobreposição entre tiles
MIN_VISIVEL = 0.5      # fração mínima da vaca dentro do tile para o rótulo valer
NEG_TILES_POR_POS = 0.5   # tiles sem vaca usados no treino por tile com vaca
NEG_IMG_FRAC = 0.5     # imagens sem vaca mantidas por imagem com vaca (por conjunto)
MODELO_BASE = "yolo11s.pt"   # yolo11n.pt (rápido), yolo11s.pt (equilíbrio), yolo11m.pt (mais preciso)
EPOCHS = 100
BATCH = 8              # reduza para 4 se faltar memória de GPU
WORKERS = 2            # se o treino travar no Windows, troque para 0
FRACAO = (0.70, 0.15, 0.15)   # treino / validação / teste, por voo
REEXTRAIR = False      # True = refaz extração e tiles mesmo se já existirem
USAR_EXTRAS = True     # False = treina só com o Zenodo (pasto aéreo)
RAPIDO = False          # True = treino curto só para validar o fluxo (yolo11n, 8 épocas, no máximo 300 imagens de treino por cenário extra). False = treino completo
PERMITIR_CPU = False   # False = para com erro se não houver GPU (em CPU o treino leva dias)
FLIPUD = 0.0          # giro vertical: válido em vista aérea, mas vira a vaca de cabeça para baixo nas fotos de solo; valor baixo é um compromisso

MAX_TREINO_EXTRA = 0
if RAPIDO:
    MODELO_BASE, EPOCHS, MAX_TREINO_EXTRA = "yolo11n.pt", 8, 300
print("Modo:", "RÁPIDO (só para validar o fluxo; os números não valem como resultado final)" if RAPIDO else "COMPLETO", "|", MODELO_BASE, "|", EPOCHS, "épocas")

# Cada execução tem pastas próprias. O app só usa data/processed após promoção explícita.
EXPERIMENTO = datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ("_rapido" if RAPIDO else "_completo")
YOLO_DIR = PROJ / "data" / "experimentos" / EXPERIMENTO / "yolo"
OUT = PROJ / "data" / "experimentos" / EXPERIMENTO / "artefatos"
for p in (YOLO_DIR, OUT):
    p.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
assert ZIP.exists(), "Rode antes: python 01_obter_dados.py (ou python 01_obter_dados.py --seletivo)"
print("Pasta do projeto:", PROJ)
print("Zip usado:", ZIP.name)
print("Extras:", EXTRAS if USAR_EXTRAS else "desligados (USAR_EXTRAS = False)", "| existe:", EXTRAS.exists())
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA (o treino em CPU vai demorar muito)")
except Exception as e:
    print("torch indisponível:", e)

## 2. Inventário do zip: imagens × rótulos

Cada imagem é pareada com o seu rótulo YOLO pelo caminho (fazenda / voo / nome). Imagem sem arquivo de rótulo ou com arquivo vazio = **sem vaca**.
Nada é extraído ainda.

In [ ]:
def parse(nome):
    # Estrutura do zip: <raiz>/<fazenda>/JPGImages/<voo>/<img>.JPG  e  <raiz>/<fazenda>/YOLO_1.1 (ou YOLO1.1)/<voo>/[obj_train_data/]<img>.txt
    partes = [p for p in nome.split("/") if p]
    if not partes or "__MACOSX" in partes or nome.endswith("/"):
        return None
    stem, _, ext = partes[-1].rpartition(".")
    if stem.startswith("~$"):
        return None
    pastas = partes[:-1]
    if "JPGImages" in pastas:
        if ext.lower() not in ("jpg", "jpeg", "png"):
            return None
        i, tipo = pastas.index("JPGImages"), "imagem"
    else:
        i = next((k for k, p in enumerate(pastas) if p.lower().startswith("yolo")), None)
        if i is None or ext.lower() != "txt" or stem.lower() == "train":   # train.txt é lista de imagens, não rótulo
            return None
        tipo = "rotulo"
    return tipo, (partes[i - 1] if i > 0 else "(raiz)"), (partes[i + 1] if len(partes) > i + 2 else "(sem voo)"), stem

imgs, rots, dups = {}, {}, []
with zipfile.ZipFile(ZIP) as z:
    for info in z.infolist():
        p = parse(info.filename)
        if p is None:
            continue
        tipo, fazenda, voo, stem = p
        alvo = imgs if tipo == "imagem" else rots
        chave = (fazenda, stem.lower())       # pareia imagem e rótulo por fazenda + nome (o nome tem data e hora)
        if chave in alvo:
            dups.append(info.filename)
        alvo[chave] = (info.filename, voo)
    assert not dups, f"{len(dups)} nomes repetidos na mesma fazenda, ex.: {dups[:3]}"
    linhas_rotulo = {k: ct.linhas_yolo_unicas(z.read(v[0]).decode("utf-8", "replace").splitlines()) for k, v in rots.items()}

inv = pd.DataFrame([{"fazenda": k[0], "voo": v[1], "stem": k[1], "membro": v[0],
                     "n_vacas": len(linhas_rotulo.get(k, []))} for k, v in imgs.items()])
inv["grupo"] = inv.fazenda + "/" + inv.voo
orfaos = set(rots) - set(imgs)
print(f"Imagens no zip: {len(inv)} | rótulos: {len(rots)} | rótulos sem imagem: {len(orfaos)}")
print(f"Imagens com vaca: {(inv.n_vacas > 0).sum()} | sem vaca: {(inv.n_vacas == 0).sum()} | caixas: {inv.n_vacas.sum()}")
print(inv.groupby("fazenda").agg(voos=("grupo", "nunique"), imagens=("stem", "size"), com_vaca=("n_vacas", lambda s: (s > 0).sum()), caixas=("n_vacas", "sum")))

In [ ]:
# VERIFICAÇÃO 1: o zip é o dataset esperado?
CAIXAS_ZENODO = 4941   # o que a página do Zenodo declara
pareados = [k for k in rots if k in imgs]
total_caixas = sum(len(linhas_rotulo[k]) for k in pareados)
sem_rotulo = set(imgs) - set(rots)
print(f"Imagens sem arquivo de rótulo: {len(sem_rotulo)}")
assert not sem_rotulo, f"{len(sem_rotulo)} imagens sem rótulo (ex.: {list(sem_rotulo)[:3]}). Não vou assumir que são 'sem vaca': confira o pareamento."
if ZIP == ZIP_COMPLETO:
    if orfaos:   # no zip completo existem 2 rótulos de Other_farms cuja imagem não está no zip; são ignorados
        print(f"Rótulos sem imagem (ignorados): {len(orfaos)} -> {[rots[k][0] for k in sorted(orfaos)]} | caixas nesses rótulos: {sum(len(linhas_rotulo[k]) for k in orfaos)}")
    print(f"Caixas nos rótulos pareados com imagem: {total_caixas} | Zenodo declara {CAIXAS_ZENODO} (diferença {CAIXAS_ZENODO - total_caixas}, {100 * (CAIXAS_ZENODO - total_caixas) / CAIXAS_ZENODO:.1f}%)")
    assert total_caixas > 0.9 * CAIXAS_ZENODO, "Muito menos caixas que o declarado: confira a leitura dos rótulos."
    assert len(inv) == 1385, f"Esperava 1385 imagens (página do Zenodo), achei {len(inv)}. Confira o inventário acima."
else:        # zip reduzido: tem todos os rótulos mas só parte das imagens; os rótulos das imagens não baixadas são esperados
    print(f"Zip reduzido: {len(imgs)} imagens com rótulo | {len(orfaos)} rótulos de imagens não baixadas (esperado) | caixas nas imagens baixadas: {total_caixas}")
    assert total_caixas > 0, "Nenhuma caixa nas imagens escolhidas: confira a leitura dos rótulos."
print("OK. A contagem 'real' usada no projeto é a soma dessas caixas dos arquivos YOLO (pode ficar um pouco abaixo do declarado no Zenodo).")

## 3. Classes dos rótulos

A página do dataset não lista nomes de classes. Conferimos os IDs que realmente existem nos arquivos. Todo o projeto trata como **uma classe: vaca**.

In [ ]:
with zipfile.ZipFile(ZIP) as z:
    nomes_cls = {z.read(n).decode("utf-8", "replace").strip() for n in z.namelist() if n.endswith("obj.names")}
print("Nomes de classe (obj.names):", nomes_cls)
ids = Counter(l.split()[0] for ls in linhas_rotulo.values() for l in ls)
print("IDs de classe encontrados nos rótulos:", dict(ids))
mal = [(k, l) for k, ls in linhas_rotulo.items() for l in ls if len(l.split()) != 5]
print("Linhas fora do formato 'classe cx cy w h':", len(mal))
assert set(ids) == {"0"}, f"Há mais de um ID de classe: {dict(ids)}. Decida como mapear antes de seguir (o projeto assume 1 classe)."
assert not mal, mal[:3]
print("OK: classe única (0 = vaca).")

## 4. Divisão treino / validação / teste por voo

`GroupShuffleSplit` sobre o voo (fazenda/voo). Mostra imagens e caixas de cada conjunto; se algum conjunto ficar pequeno demais, mude `SEED` e rode de novo.

In [ ]:
gss1 = GroupShuffleSplit(n_splits=1, train_size=FRACAO[0], random_state=SEED)
i_tr, i_resto = next(gss1.split(inv, groups=inv.grupo))
resto = inv.iloc[i_resto]
gss2 = GroupShuffleSplit(n_splits=1, train_size=FRACAO[1] / (FRACAO[1] + FRACAO[2]), random_state=SEED)
i_va, i_te = next(gss2.split(resto, groups=resto.grupo))
inv["conjunto"] = ""
inv.loc[inv.index[i_tr], "conjunto"] = "train"
inv.loc[resto.index[i_va], "conjunto"] = "val"
inv.loc[resto.index[i_te], "conjunto"] = "test"
res = inv.groupby("conjunto").agg(voos=("grupo", "nunique"), imagens=("stem", "size"), com_vaca=("n_vacas", lambda s: (s > 0).sum()), caixas=("n_vacas", "sum"))
print(res)
# VERIFICAÇÃO 2: nenhum voo em dois conjuntos; conjuntos com vacas suficientes
assert inv.groupby("grupo").conjunto.nunique().max() == 1, "Vazamento: um voo caiu em dois conjuntos"
assert (inv.conjunto != "").all()
assert (res.caixas >= 100).all(), "Algum conjunto tem menos de 100 caixas: mude SEED e rode de novo"
print("OK: divisão por voo sem vazamento.")

## 5. Selecionar e extrair as imagens

Ficam **todas** as imagens com vaca e uma parte das sem vaca (`NEG_IMG_FRAC`) para medir falsos positivos. O zip de 16,6 GB **não** é extraído por inteiro: só as imagens escolhidas.

In [ ]:
rng = np.random.default_rng(SEED)
inv["usar"] = inv.n_vacas > 0
for cj, g in inv.groupby("conjunto"):
    neg = g[g.n_vacas == 0].index.to_numpy()
    n = int(round(NEG_IMG_FRAC * (g.n_vacas > 0).sum()))
    if len(neg):
        inv.loc[rng.choice(neg, size=min(n, len(neg)), replace=False), "usar"] = True
sel = inv[inv.usar].copy()
sel["arquivo"] = [f"{r.fazenda}__{r.voo.replace('/', '_')}__{r.stem}" for r in sel.itertuples()]
print(sel.groupby("conjunto").agg(imagens=("stem", "size"), com_vaca=("n_vacas", lambda s: (s > 0).sum()), caixas=("n_vacas", "sum")))

pasta = lambda cj: INTERIM / cj
if REEXTRAIR and INTERIM.exists():
    shutil.rmtree(INTERIM); INTERIM.mkdir(parents=True)
with zipfile.ZipFile(ZIP) as z:
    for r in sel.itertuples():
        destino = pasta(r.conjunto) / "imagens" / f"{r.arquivo}.jpg"
        destino.parent.mkdir(parents=True, exist_ok=True)
        if not destino.exists():
            destino.write_bytes(z.read(r.membro))
        (pasta(r.conjunto) / "rotulos").mkdir(parents=True, exist_ok=True)
        (pasta(r.conjunto) / "rotulos" / f"{r.arquivo}.txt").write_text("\n".join(linhas_rotulo.get((r.fazenda, r.stem), [])), encoding="utf-8")
print("Extraído em", INTERIM)

In [ ]:
# VERIFICAÇÃO 3: imagens legíveis, tamanhos e tamanho típico da vaca em pixels
amostra = sel.sample(min(60, len(sel)), random_state=SEED)
tamanhos, lados = Counter(), []
for r in amostra.itertuples():
    im = cv2.imread(str(pasta(r.conjunto) / "imagens" / f"{r.arquivo}.jpg"))
    assert im is not None, f"imagem ilegível: {r.arquivo}"
    h, w = im.shape[:2]; tamanhos[(w, h)] += 1
    for a, b, c, d in ct.yolo_para_pixels(linhas_rotulo.get((r.fazenda, r.stem), []), w, h):
        lados.append((c - a, d - b))
print("Tamanhos de imagem (largura, altura):", dict(tamanhos))
lados = np.array(lados)
assert len(lados) > 0
print(f"Caixa da vaca (px) — mediana {np.median(lados[:,0]):.0f}×{np.median(lados[:,1]):.0f} | P5 {np.percentile(lados.max(1),5):.0f} | P95 {np.percentile(lados.max(1),95):.0f} (maior lado)")

## 6. Gerar os tiles no formato YOLO

- **train e val:** tiles de `TILE` px com sobreposição. Entram todos os tiles com vaca e uma amostra de tiles vazios (`NEG_TILES_POR_POS` por tile positivo). A vaca só conta no tile se ao menos `MIN_VISIVEL` dela estiver dentro.
- **test:** também em tiles (para o mAP por tile), mas a **contagem é avaliada nas imagens inteiras** na seção 9.

In [ ]:
def gerar_tiles(conjunto):
    dst_i, dst_l = YOLO_DIR / "images" / conjunto, YOLO_DIR / "labels" / conjunto
    if REEXTRAIR:
        shutil.rmtree(dst_i, ignore_errors=True); shutil.rmtree(dst_l, ignore_errors=True)
    dst_i.mkdir(parents=True, exist_ok=True); dst_l.mkdir(parents=True, exist_ok=True)
    pos = neg = caixas = 0
    for r in sel[sel.conjunto == conjunto].itertuples():
        im = cv2.imread(str(pasta(conjunto) / "imagens" / f"{r.arquivo}.jpg"))
        h, w = im.shape[:2]
        cx = ct.yolo_para_pixels(linhas_rotulo.get((r.fazenda, r.stem), []), w, h)
        positivos, negativos = [], []
        for x0, y0 in ct.grade_tiles(w, h, TILE, OVERLAP):
            tw, th = min(TILE, w - x0), min(TILE, h - y0)
            if ct.tile_tem_fragmento_sem_rotulo(cx, x0, y0, tw, th, MIN_VISIVEL):
                continue   # não mostrar animal sem caixa como exemplo de fundo
            lin = ct.rotulos_do_tile(cx, x0, y0, tw, th, MIN_VISIVEL)
            (positivos if lin else negativos).append((x0, y0, tw, th, lin))
        n_neg = int(round(NEG_TILES_POR_POS * len(positivos))) if positivos else 1
        if len(negativos) > n_neg:
            negativos = [negativos[i] for i in rng.choice(len(negativos), size=n_neg, replace=False)]
        for x0, y0, tw, th, lin in positivos + negativos:
            nome = f"{r.arquivo}__x{x0}_y{y0}"
            if not (dst_i / f"{nome}.jpg").exists():
                cv2.imwrite(str(dst_i / f"{nome}.jpg"), im[y0:y0 + th, x0:x0 + tw], [cv2.IMWRITE_JPEG_QUALITY, 95])
            (dst_l / f"{nome}.txt").write_text("\n".join(lin), encoding="utf-8")
        pos += len(positivos); neg += len(negativos); caixas += sum(len(t[4]) for t in positivos)
    print(f"{conjunto:5s}: {pos} tiles com vaca + {neg} vazios | {caixas} caixas nos tiles")

for cj in ("train", "val", "test"):
    gerar_tiles(cj)
yaml = YOLO_DIR / "dataset.yaml"
yaml.write_text(f"path: {YOLO_DIR.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: vaca\n", encoding="utf-8")
print("Config:", yaml)

## 6b. Dados extras: Open Images e COLO (modelo geral)

Vêm de `python 03_obter_dados_extras.py` (Open Images e COLO) e de `python 04_obter_cattleeyeview.py` (CattleEyeView, opcional), já em YOLO, com classe única. **Não usam tiles:** são fotos de até 1.024 px (Open Images) ou 1.152 px (COLO) e a vaca é grande. Entram inteiras no treino e na contagem.

**Divisão sem vazamento**
- **Open Images:** divisão oficial (train / validation / test).
- **COLO:** a divisão train/test dos autores **não** é usada. Nos arquivos baixados, `top_<n>` e `side_<n>` compartilham o mesmo `<n>` entre train e test (a mesma cena aparece nos dois lados) e no `external` há sequências repetidas nos dois lados. A divisão é refeita **por grupo**: cena `<n>` para topo + lateral, sequência `<n>` para o externo. Por isso os números de COLO daqui **não** são comparáveis com os do artigo.

- **CattleEyeView:** divisão dos autores por vídeo (teste = vídeos 01, 05, 07, 09, 10; a pasta de validação deles repete o teste e não é usada). A validação sai de 2 vídeos de treino sorteados com a `SEED`. Quadros do mesmo vídeo são muito parecidos entre si, por isso o grupo é o vídeo.

Cada fonte/ângulo é um **cenário** (`dominio`), e o erro de contagem é medido por cenário na seção 9.

In [ ]:
# 6b-1. Inventário dos dados extras e divisão por grupo
NOMES_DOMINIO = {"aereo_pasto": "Drone vertical em pasto (Zenodo)",
                 "colo_topo": "Vista de cima em instalação (COLO)",
                 "colo_lateral": "Vista lateral em instalação (COLO)",
                 "colo_externo": "Vista externa (COLO)",
                 "cev_rampa": "Vista de cima em rampa de curral (CattleEyeView)",
                 "foto_openimages": "Foto comum (Open Images)",
                 "brasil_solo": "Rebanho brasileiro de solo (revisado)"}
DOM_COLO = {"1_top": "colo_topo", "2_side": "colo_lateral", "3_external": "colo_externo"}
COLS_EXT = ["dominio", "fonte", "subconjunto", "stem", "img", "rot", "n_vacas", "grupo", "conjunto"]
seguro = lambda s: re.sub(r"[^A-Za-z0-9_-]", "_", s)
contar_linhas = lambda p: len(ct.linhas_yolo_unicas(p.read_text(encoding="utf-8").splitlines()))


def grupo_colo(nome, cfg):
    """Grupo de um arquivo do COLO (ex.: top_100_4_jpg.rf.<hash>): cena <n> para topo + lateral, sequência <n> para o externo."""
    m = re.match(r"(?:top|side|img)_(\d+)(?:_\d+)?_jpg\.rf\.", nome)
    assert m, f"nome fora do padrão do COLO: {nome}"
    return ("ext" if cfg == "3_external" else "cena") + m.group(1)


def dividir_grupos(df, fracao, seed, tentativas=2000):
    """Atribui train/val/test por grupo (nenhum grupo em dois conjuntos). Grupos têm tamanhos muito desiguais (no COLO externo, 4 grupos
    têm mais da metade das imagens), então testa várias ordens e fica com a que mais se aproxima das frações em número de imagens."""
    tam = df.groupby("grupo").size()
    grupos, n = list(tam.index), tam.sum()
    alvo = np.array(fracao) * n
    melhor, melhor_erro = None, np.inf
    for k in range(tentativas):
        ordem = list(grupos)
        np.random.default_rng(seed + k).shuffle(ordem)
        corte1, corte2 = alvo[0], alvo[0] + alvo[1]
        destino, acumulado = {}, 0
        for g in ordem:
            destino[g] = "train" if acumulado < corte1 else ("val" if acumulado < corte2 else "test")
            acumulado += tam[g]
        tot = pd.Series({g: tam[g] for g in grupos}).groupby(pd.Series(destino)).sum().reindex(["train", "val", "test"]).fillna(0).values
        erro = np.abs(tot - alvo).sum()
        if (tot > 0).all() and erro < melhor_erro:
            melhor, melhor_erro = destino, erro
    assert melhor is not None, "não foi possível dividir os grupos em três conjuntos"
    return df.grupo.map(melhor)


registros = []
if USAR_EXTRAS and EXTRAS.exists():
    for split_oi, cj in (("train", "train"), ("validation", "val"), ("test", "test")):          # Open Images: divisão oficial
        for img in sorted((EXTRAS / "openimages" / "images" / split_oi).glob("*.jpg")):
            rot = EXTRAS / "openimages" / "labels" / split_oi / f"{img.stem}.txt"
            assert rot.exists(), f"sem rótulo: {img}"
            registros.append(dict(dominio="foto_openimages", fonte="OpenImages", subconjunto=split_oi, stem=img.stem, img=img, rot=rot,
                                  n_vacas=contar_linhas(rot), grupo=f"oi/{img.stem}", conjunto=cj))
    colo = []
    for cfg in DOM_COLO:
        for sp in ("train", "test"):                       # só para ler os arquivos; o conjunto é decidido abaixo, por grupo
            for img in sorted((EXTRAS / "colo" / "images" / f"{cfg}_{sp}").glob("*.jpg")):
                rot = EXTRAS / "colo" / "labels" / f"{cfg}_{sp}" / f"{img.stem}.txt"
                assert rot.exists(), f"sem rótulo: {img}"
                colo.append(dict(dominio=DOM_COLO[cfg], fonte="COLO", subconjunto=cfg, stem=img.stem, img=img, rot=rot,
                                 n_vacas=contar_linhas(rot), grupo=grupo_colo(img.stem, cfg),
                                 familia="ext" if cfg == "3_external" else "cena"))
    colo = pd.DataFrame(colo)
    if len(colo):
        assert colo.stem.is_unique, "nomes repetidos no COLO"
        colo["conjunto"] = ""
        for fam, g in colo.groupby("familia"):             # divide cada família separadamente, para as duas ficarem proporcionais
            colo.loc[g.index, "conjunto"] = dividir_grupos(g, FRACAO, SEED)
        registros += colo.drop(columns="familia").to_dict("records")
    cev_dir = EXTRAS / "cattleeyeview"
    if (cev_dir / "inventario.csv").exists():           # CattleEyeView: divisão dos autores por vídeo (a pasta val deles repete a test e não é usada)
        cev = pd.read_csv(cev_dir / "inventario.csv", dtype={"video": str})
        videos_treino = sorted(cev[cev.split_autores == "train"].video.unique())
        val_v = set(np.random.default_rng(SEED).choice(videos_treino, size=min(2, max(len(videos_treino) - 1, 0)), replace=False))
        for r in cev.itertuples():
            img_ = cev_dir / "images" / f"v{r.video}" / f"{r.stem}.jpg"
            rot_ = cev_dir / "labels" / f"v{r.video}" / f"{r.stem}.txt"
            assert img_.exists() and rot_.exists(), f"arquivo do CattleEyeView ausente: {img_.name}. Rode de novo: python 04_obter_cattleeyeview.py"
            cj_ = "test" if r.split_autores == "test" else ("val" if r.video in val_v else "train")
            registros.append(dict(dominio="cev_rampa", fonte="CattleEyeView", subconjunto=f"v{r.video}", stem=r.stem, img=img_, rot=rot_,
                                  n_vacas=int(r.n_vacas), grupo=f"cev/{r.video}", conjunto=cj_))
from dados_locais import carregar_revisados
registros += carregar_revisados(PROJ).to_dict("records")
ext = pd.DataFrame(registros, columns=COLS_EXT) if registros else pd.DataFrame(columns=COLS_EXT)
ext["arquivo"] = [f"{d}__{seguro(s)}" for d, s in zip(ext.dominio, ext.stem)]
if MAX_TREINO_EXTRA and len(ext):          # modo rápido: menos imagens de treino por cenário (validação e teste ficam completos)
    manter = [i for _, g in ext[ext.conjunto == "train"].groupby("dominio") for i in g.sample(min(len(g), MAX_TREINO_EXTRA), random_state=SEED).index]
    ext = ext[(ext.conjunto != "train") | ext.index.isin(manter)].reset_index(drop=True)
    print(f"Modo rápido: treino dos extras limitado a {MAX_TREINO_EXTRA} imagens por cenário.")

if len(ext):
    assert ext.arquivo.is_unique, "nomes de arquivo repetidos entre as fontes extras"
    res_ext = ext.groupby(["dominio", "conjunto"]).agg(imagens=("stem", "size"), com_vaca=("n_vacas", lambda s: int((s > 0).sum())), caixas=("n_vacas", "sum"))
    print(res_ext)
    # VERIFICAÇÃO 2b: nenhum grupo em dois conjuntos; cada cenário tem imagens e caixas suficientes em validação e teste
    assert ext.groupby("grupo").conjunto.nunique().max() == 1, "Vazamento: um grupo caiu em dois conjuntos"
    assert (ext.conjunto.isin(["train", "val", "test"])).all()
    for d in ext.dominio.unique():
        for cj in ("val", "test"):
            g = ext[(ext.dominio == d) & (ext.conjunto == cj)]
            assert len(g) >= 20 and g.n_vacas.sum() >= 100, f"{d}/{cj}: só {len(g)} imagens e {g.n_vacas.sum()} caixas. Mude SEED e rode de novo."
    print("OK: dados extras divididos sem vazamento.")
else:
    print("Sem dados extras (USAR_EXTRAS = False ou pasta ausente): treino só com o Zenodo.")

In [ ]:
# 6b-2. Colocar os extras na pasta do YOLO, junto com os tiles (hardlink; se não der, cópia)
PREFIXOS = tuple(f"{d}__" for d in NOMES_DOMINIO if d != "aereo_pasto")
for cj in ("train", "val", "test"):          # limpa extras de execuções anteriores: a divisão pode ter mudado
    for pasta_ in (YOLO_DIR / "images" / cj, YOLO_DIR / "labels" / cj):
        if pasta_.exists():
            for p in pasta_.iterdir():
                if p.name.startswith(PREFIXOS):
                    p.unlink()

def ligar(origem, destino):
    try:
        os.link(origem, destino)
    except OSError:
        shutil.copy2(origem, destino)

for r in ext.itertuples():
    di, dl = YOLO_DIR / "images" / r.conjunto, YOLO_DIR / "labels" / r.conjunto
    di.mkdir(parents=True, exist_ok=True); dl.mkdir(parents=True, exist_ok=True)
    ligar(r.img, di / f"{r.arquivo}.jpg")
    (dl / f"{r.arquivo}.txt").write_text("\n".join(ct.linhas_yolo_unicas(Path(r.rot).read_text(encoding="utf-8").splitlines())), encoding="utf-8")
for cj in ("train", "val", "test"):
    todos = list((YOLO_DIR / "images" / cj).glob("*.jpg"))
    n_ext = sum(p.name.startswith(PREFIXOS) for p in todos)
    print(f"{cj:5s}: {len(todos) - n_ext} tiles do Zenodo + {n_ext} imagens inteiras dos extras")

In [ ]:
# VERIFICAÇÃO 4: todo tile tem rótulo; nenhum rótulo fora de [0,1]
for cj in ("train", "val", "test"):
    ims = {p.stem for p in (YOLO_DIR / "images" / cj).glob("*.jpg")}
    lbs = {p.stem for p in (YOLO_DIR / "labels" / cj).glob("*.txt")}
    assert ims == lbs, f"{cj}: imagens e rótulos não batem"
    for p in (YOLO_DIR / "labels" / cj).glob("*.txt"):
        for l in p.read_text().splitlines():
            v = [float(x) for x in l.split()[1:]]
            assert all(0 <= x <= 1 for x in v), (p.name, l)
print("OK: tiles e rótulos consistentes.")
# ver 6 tiles de treino com as caixas
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
ps = [p for p in sorted((YOLO_DIR / "labels" / "train").glob("*.txt")) if p.stat().st_size > 0]
for a, p in zip(ax.ravel(), random.sample(ps, min(6, len(ps)))):
    im = cv2.cvtColor(cv2.imread(str(YOLO_DIR / "images" / "train" / f"{p.stem}.jpg")), cv2.COLOR_BGR2RGB)
    for q in ct.yolo_para_pixels(p.read_text().splitlines(), im.shape[1], im.shape[0]).astype(int):
        cv2.rectangle(im, (int(q[0]), int(q[1])), (int(q[2]), int(q[3])), (0, 255, 0), 3)
    a.imshow(im); a.axis("off"); a.set_title(p.stem[-40:], fontsize=8)
plt.tight_layout(); plt.show()

## 7. Treinar o YOLO

Começa dos pesos pré-treinados (`MODELO_BASE`, baixados automaticamente na primeira vez). Os melhores pesos (pela validação) ficam em `runs/gado/treino/weights/best.pt`.
Pode interromper e retomar: `YOLO("runs/gado/treino/weights/last.pt").train(resume=True)`.

In [ ]:
from ultralytics import YOLO
import torch

assert torch.cuda.is_available() or PERMITIR_CPU, (
    "GPU CUDA não disponível: o treino rodaria em CPU. Confira `python 00_verificar_ambiente.py` e a versão do torch (RTX 50xx exige build cu128 ou mais novo). "
    "Para treinar em CPU mesmo assim, PERMITIR_CPU = True.")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Treinando em:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "| torch", torch.__version__, "| CUDA", torch.version.cuda)
modelo = YOLO(MODELO_BASE)
res_treino = modelo.train(data=str(yaml), imgsz=TILE, epochs=EPOCHS, batch=BATCH, workers=WORKERS, device=DEVICE, seed=SEED,
                          project=str(PROJ / "runs" / "gado"), name=EXPERIMENTO, exist_ok=False, patience=20,
                          flipud=FLIPUD, fliplr=0.5, degrees=0, mosaic=1.0, plots=True)
BEST = Path(res_treino.save_dir) / "weights" / "best.pt"
assert BEST.exists(), "best.pt não foi gerado"
print("Melhores pesos:", BEST)

In [ ]:
hist = pd.read_csv(Path(res_treino.save_dir) / "results.csv"); hist.columns = [c.strip() for c in hist.columns]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(hist["epoch"], hist["train/box_loss"], label="treino"); ax[0].plot(hist["epoch"], hist["val/box_loss"], label="validação")
ax[0].set(title="Perda de caixa", xlabel="época"); ax[0].legend()
ax[1].plot(hist["epoch"], hist["metrics/mAP50(B)"], label="mAP50"); ax[1].plot(hist["epoch"], hist["metrics/mAP50-95(B)"], label="mAP50-95")
ax[1].set(title="Validação por tile", xlabel="época"); ax[1].legend()
plt.tight_layout(); plt.show()
hist.tail(3)

## 8. Métricas por tile (validação e teste)

Mede detecção por tile (Zenodo) e por imagem (COLO e Open Images), todas as fontes juntas. É útil para comparar treinos, mas **não é** a medida de contagem.

In [ ]:
modelo = YOLO(str(BEST))
m_val = modelo.val(data=str(yaml), split="val", imgsz=TILE, batch=BATCH, device=DEVICE, plots=False, verbose=False)
m_te = modelo.val(data=str(yaml), split="test", imgsz=TILE, batch=BATCH, device=DEVICE, plots=False, verbose=False)
tab = pd.DataFrame({"conjunto": ["validação", "teste"],
                    "precisao": [m_val.box.mp, m_te.box.mp], "recall": [m_val.box.mr, m_te.box.mr],
                    "mAP50": [m_val.box.map50, m_te.box.map50], "mAP50_95": [m_val.box.map, m_te.box.map]}).round(3)
print(tab.to_string(index=False))

## 9. Contagem em imagens inteiras, por cenário

Cada imagem de validação e de teste é detectada com o modo do seu cenário: **tiles** com junção de duplicatas (drone) ou **imagem inteira** (COLO e Open Images). Limiar de confiança baixo (0,05) na detecção; o limiar final é aplicado depois.

- **Limiar por cenário:** o de menor erro de contagem (MAE) na **validação** daquele cenário.
- **Limiar global:** o de menor erro relativo médio (WAPE) entre os cenários, na validação. Só serve de padrão quando o tipo de imagem é desconhecido.
- O **teste** só é olhado depois, uma vez, com os limiares da validação.


In [ ]:

# 9.1 Todas as imagens inteiras, com o cenário e o modo de detecção de cada uma
zen = sel.copy()
zen["dominio"], zen["fonte"], zen["subconjunto"], zen["modo"] = "aereo_pasto", "Zenodo", zen.voo, "tiles"
zen["img"] = [pasta(r.conjunto) / "imagens" / f"{r.arquivo}.jpg" for r in zen.itertuples()]
zen["rot"] = [pasta(r.conjunto) / "rotulos" / f"{r.arquivo}.txt" for r in zen.itertuples()]
ext_ = ext.copy()
ext_["fazenda"], ext_["voo"], ext_["modo"] = ext_.fonte, ext_.subconjunto, "inteira"
COLS = ["arquivo", "dominio", "fonte", "fazenda", "voo", "grupo", "stem", "conjunto", "n_vacas", "img", "rot", "modo"]
cat = pd.concat([zen[COLS], ext_[COLS]], ignore_index=True)
assert cat.arquivo.is_unique
DOMINIOS = [d for d in NOMES_DOMINIO if d in set(cat[cat.conjunto == "val"].dominio) & set(cat[cat.conjunto == "test"].dominio)]
print(cat[cat.conjunto.isin(["val", "test"])].groupby(["dominio", "conjunto"]).agg(imagens=("arquivo", "size"), caixas=("n_vacas", "sum")).unstack(fill_value=0))


In [ ]:
# 9.2 Detectar apenas VALIDAÇÃO na grade; nenhuma escolha olha o teste.
IMGSZ_GRADE = (1024, 1280, 1536)
IOU_GRADE = (0.5, 0.7, 0.85)

referencias = {}

def detectar_conjunto(cj):
    saida = {}
    for r in cat[cat.conjunto == cj].itertuples():
        img = cv2.imread(str(r.img))
        assert img is not None, f"imagem ilegível: {r.img}"
        h, w = img.shape[:2]
        referencias[r.arquivo] = ct.yolo_para_pixels(Path(r.rot).read_text(encoding="utf-8").splitlines(), w, h)
        tamanhos = (TILE,) if r.modo == "tiles" else (IMGSZ_GRADE if cj == "val" else (IMGSZ_DOM[r.dominio],))
        ious = IOU_GRADE if cj == "val" else (IOU_DOM[r.dominio],)
        for sz in tamanhos:
            for iou in ious:
                if r.modo == "tiles":
                    bs = ct.detectar(modelo, img, tile=TILE, overlap=OVERLAP, iou_local=iou, device=DEVICE)
                else:
                    bs = ct.detectar_inteira(modelo, img, imgsz=sz, iou_nms=iou, device=DEVICE)
                saida[(r.arquivo, sz, iou)] = bs
    return saida

det = {"val": detectar_conjunto("val")}
real = {cj: cat[cat.conjunto == cj].set_index("arquivo") for cj in ("val", "test")}


In [ ]:
# 9.3 Métricas de contagem e escolha de tamanho de entrada (imgsz) e limiar na VALIDAÇÃO, por cenário
def contar_dom(cj, d, sz, iou, conf):
    """Contagem por imagem de um cenário, com um tamanho de entrada e um limiar."""
    idx = real[cj].index[real[cj].dominio == d]
    return pd.Series({a: len(ct.filtrar(*det[cj][(a, sz, iou)], conf)[0]) for a in idx})

from avaliacao import metricas_contagem as metricas, comparar_caixas, taxas_deteccao

GRADE = np.round(np.arange(0.10, 0.91, 0.05), 2)
linhas = []
for d in DOMINIOS:
    modo_d = str(cat[cat.dominio == d].modo.iloc[0])
    y = real["val"].loc[real["val"].dominio == d, "n_vacas"]
    for sz in ((TILE,) if modo_d == "tiles" else IMGSZ_GRADE):
        for iou in IOU_GRADE:
            for conf in GRADE:
                p = contar_dom("val", d, sz, iou, conf).reindex(y.index)
                pares = [comparar_caixas(referencias[a], *ct.filtrar(*det["val"][(a, sz, iou)], conf)) for a in y.index]
                tp, fp, fn = (sum(m[k] for m in pares) for k in ("TP", "FP", "FN"))
                linhas.append(dict(dominio=d, imgsz=int(sz), iou=float(iou), conf=float(conf), **metricas(y, p),
                                   **taxas_deteccao(tp, fp, fn), TP=tp, FP=fp, FN=fn))
varredura = pd.DataFrame(linhas)

# melhor combinação por cenário: menor MAE na validação; empate -> menor tamanho, depois menor limiar
melhor = {d: g.sort_values(["MAE", "F1_caixas", "imgsz", "iou", "conf"], ascending=[True, False, True, True, True]).iloc[0] for d, g in varredura.groupby("dominio")}
IMGSZ_DOM = {d: int(m.imgsz) for d, m in melhor.items()}
CONF_DOM = {d: float(m.conf) for d, m in melhor.items()}
IOU_DOM = {d: float(m.iou) for d, m in melhor.items()}

# limiar global: cada cenário no seu tamanho; menor média dos WAPE entre cenários
vg = varredura[(varredura.imgsz == varredura.dominio.map(IMGSZ_DOM)) & (varredura.iou == varredura.dominio.map(IOU_DOM))]
macro_wape = vg.groupby("conf").WAPE_pct.mean()
CONF_GLOBAL = float(macro_wape.idxmin())
print(varredura.pivot_table(index=["imgsz", "iou", "conf"], columns="dominio", values="MAE").round(2).to_string())
print("\nTamanho de entrada por cenário (menor MAE na validação):", IMGSZ_DOM)
print("Limiar por cenário:", CONF_DOM)
print("Limiar global (menor WAPE médio entre cenários):", CONF_GLOBAL)

print("NMS por cenário:", IOU_DOM)


In [ ]:

# VERIFICAÇÃO 5: a contagem por imagem soma as caixas e o teste usa tamanho e limiar escolhidos na validação
det["test"] = detectar_conjunto("test")
te = real["test"].copy()
te["conf_usada"] = te.dominio.map(CONF_DOM)
te["imgsz_usado"] = te.dominio.map(IMGSZ_DOM)
assert te.conf_usada.notna().all() and te.imgsz_usado.notna().all(), "há cenário no teste sem configuração da validação"
pt = pd.Series({a: len(ct.filtrar(*det["test"][(a, int(te.loc[a, "imgsz_usado"]), IOU_DOM[te.loc[a, "dominio"]])], te.loc[a, "conf_usada"])[0]) for a in te.index})
pt_glob = pd.Series({a: len(ct.filtrar(*det["test"][(a, int(te.loc[a, "imgsz_usado"]), IOU_DOM[te.loc[a, "dominio"]])], CONF_GLOBAL)[0]) for a in te.index})
assert len(pt) == len(te) and pt.notna().all() and pt_glob.notna().all()
contagem_teste = pd.DataFrame({"arquivo": te.index, "dominio": te.dominio.values, "fazenda": te.fazenda.values, "voo": te.voo.values,
                               "real": te.n_vacas.values, "previsto": pt.reindex(te.index).values, "imgsz_usado": te.imgsz_usado.values.astype(int), "conf_usada": te.conf_usada.values,
                               "previsto_conf_global": pt_glob.reindex(te.index).values})
contagem_teste["erro"] = contagem_teste.previsto - contagem_teste.real
por_dominio = pd.DataFrame({d: metricas(g.real, g.previsto) for d, g in contagem_teste.groupby("dominio")}).T.astype(float)
por_dominio.insert(0, "iou", pd.Series(IOU_DOM))
por_dominio.insert(0, "conf", pd.Series(CONF_DOM))
por_dominio.insert(0, "imgsz", pd.Series(IMGSZ_DOM))
mt = metricas(contagem_teste.real, contagem_teste.previsto)     # todos os cenários juntos (cada um com o seu limiar)
mt["conf"] = CONF_GLOBAL
print(por_dominio.round(2).to_string())
print("\nTodos os cenários juntos:", {k: round(v, 3) for k, v in mt.items()})
print("Total real:", int(contagem_teste.real.sum()), "| total previsto:", int(contagem_teste.previsto.sum()))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
mx = max(contagem_teste.real.max(), contagem_teste.previsto.max()) + 2
for d, g in contagem_teste.groupby("dominio"):
    ax[0].scatter(g.real, g.previsto, alpha=.6, label=NOMES_DOMINIO[d])
ax[0].plot([0, mx], [0, mx], "k--", lw=1); ax[0].legend(fontsize=8)
ax[0].set(xlabel="Contagem real", ylabel="Contagem prevista", title=f"Teste: MAE {mt['MAE']:.2f} | viés {mt['vies']:+.2f} | R² {mt['R2']:.2f}")
for d, g in vg.groupby("dominio"):
    ax[1].plot(g.conf, g.WAPE_pct, marker="o", ms=3, label=NOMES_DOMINIO[d])
ax[1].plot(macro_wape.index, macro_wape.values, "k-", lw=2, label="média (limiar global)"); ax[1].axvline(CONF_GLOBAL, color="r", ls="--")
ax[1].set(xlabel="Limiar de confiança", ylabel="Erro relativo na validação (WAPE, %)", title="Escolha dos limiares (cada cenário no seu tamanho)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# Correspondência individual: separa omissões e falsos positivos que se compensam no total.
from avaliacao import comparar_caixas, taxas_deteccao
caixas_teste = []
for r in cat[cat.conjunto == "test"].itertuples():
    img = cv2.imread(str(r.img)); h, w = img.shape[:2]
    ref = ct.yolo_para_pixels(Path(r.rot).read_text(encoding="utf-8").splitlines(), w, h)
    b, s = ct.filtrar(*det["test"][(r.arquivo, IMGSZ_DOM[r.dominio], IOU_DOM[r.dominio])], CONF_DOM[r.dominio])
    caixas_teste.append(dict(arquivo=r.arquivo, dominio=r.dominio, **comparar_caixas(ref, b, s)))
caixas_teste = pd.DataFrame(caixas_teste)
caixas_teste.to_csv(OUT / "deteccao_teste.csv", index=False)
print(caixas_teste.groupby("dominio")[["TP", "FP", "FN"]].sum())


## 10. Exemplos visuais (verde = rótulo real, roxo = modelo)

Dois piores casos e um aleatório **por cenário**. Olhe os piores: é onde o modelo erra e onde vale investigar (sombra, vaca deitada, cerca, vaca cortada na borda). No COLO, parte do "erro" é vaca visível sem caixa no rótulo.


In [ ]:

EX = OUT / "exemplos"
shutil.rmtree(EX, ignore_errors=True); EX.mkdir(parents=True)
cat_i = cat.set_index("arquivo")
partes = []
for d, g in contagem_teste.groupby("dominio"):
    g = g.reindex(g.erro.abs().sort_values(ascending=False).index)
    resto_ = g.iloc[2:]
    partes.append(pd.concat([g.head(2), resto_.sample(min(1, len(resto_)), random_state=SEED)]))
escolha = pd.concat(partes).drop_duplicates("arquivo")
meta_ex = []
for r in escolha.itertuples():
    ln = cat_i.loc[r.arquivo]
    im = cv2.imread(str(ln.img))
    h, w = im.shape[:2]
    ref = ct.yolo_para_pixels(Path(ln.rot).read_text(encoding="utf-8").splitlines(), w, h)
    b, s = ct.filtrar(*det["test"][(r.arquivo, int(r.imgsz_usado), IOU_DOM[r.dominio])], r.conf_usada)
    vis = ct.desenhar(im, b, ref=ref)
    vis = cv2.resize(vis, (1600, int(1600 * h / w)), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(EX / f"{r.arquivo}.jpg"), vis, [cv2.IMWRITE_JPEG_QUALITY, 85])
    meta_ex.append(dict(arquivo=f"{r.arquivo}.jpg", dominio=r.dominio, fazenda=r.fazenda, voo=r.voo, real=int(r.real), previsto=int(r.previsto)))
pd.DataFrame(meta_ex).to_csv(OUT / "exemplos.csv", index=False)
mostrar = meta_ex[:6]
fig, ax = plt.subplots(2, 3, figsize=(18, 11))
for a, m in zip(ax.ravel(), mostrar):
    a.imshow(cv2.cvtColor(cv2.imread(str(EX / m["arquivo"])), cv2.COLOR_BGR2RGB)); a.axis("off")
    a.set_title(f"{NOMES_DOMINIO[m['dominio']]}\nreal {m['real']} × previsto {m['previsto']}", fontsize=9)
plt.tight_layout(); plt.show()


## 11. Salvar para o app

In [ ]:

def limpar(x):
    """Converte para tipos do JSON (NaN vira None)."""
    if isinstance(x, dict):
        return {k: limpar(v) for k, v in x.items()}
    if isinstance(x, (float, np.floating)):
        return None if np.isnan(x) else float(x)
    if isinstance(x, np.integer):
        return int(x)
    return x

shutil.copy(BEST, OUT / "modelo_gado.pt")
cat.to_csv(OUT / "catalogo.csv", index=False)
div_zen = inv[["fazenda", "voo", "stem", "n_vacas", "conjunto", "usar"]].assign(dominio="aereo_pasto")
div_ext = ext_[["fazenda", "voo", "stem", "n_vacas", "conjunto", "dominio"]].assign(usar=True)
pd.concat([div_zen, div_ext], ignore_index=True).to_csv(OUT / "divisao.csv", index=False)
varredura.to_csv(OUT / "varredura_conf.csv", index=False)
contagem_teste.to_csv(OUT / "contagem_teste.csv", index=False)
hist.to_csv(OUT / "curvas_treino.csv", index=False)
meta = dict(gerado_em=datetime.now().isoformat(timespec="seconds"), modelo_base=MODELO_BASE, epocas_pedidas=EPOCHS, epocas_rodadas=int(hist.epoch.max()),
            versao_inferencia=ct.VERSAO_INFERENCIA, iou_por_dominio=IOU_DOM, iou_juncao=0.5,
            tile=TILE, overlap=OVERLAP, flipud=FLIPUD, rapido=bool(RAPIDO), conf_otima=CONF_GLOBAL, conf_por_dominio=CONF_DOM, imgsz_por_dominio=IMGSZ_DOM, imgsz_grade=list(IMGSZ_GRADE), seed=SEED,
            modo_por_dominio={d: str(cat[cat.dominio == d].modo.iloc[0]) for d in DOMINIOS},
            imagens_total=int(len(inv) + len(ext)), caixas_total=int(inv.n_vacas.sum() + ext.n_vacas.sum()),
            fontes={d: dict(nome=NOMES_DOMINIO[d], imagens=int((cat.dominio == d).sum()), caixas=int(cat[cat.dominio == d].n_vacas.sum())) for d in NOMES_DOMINIO if (cat.dominio == d).any()},
            conjuntos={cj: dict(imagens_usadas=int((cat.conjunto == cj).sum()), caixas=int(cat[cat.conjunto == cj].n_vacas.sum()),
                                voos=int(cat[cat.conjunto == cj].grupo.nunique())) for cj in ("train", "val", "test")},
            tile_metricas=limpar(tab.set_index("conjunto").to_dict("index")),
            contagem_teste=limpar(mt), por_dominio=limpar(por_dominio.to_dict("index")))
json.dump(meta, open(OUT / "metricas.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print("Salvo em", OUT)
print("Artefatos isolados. Para revisar no app: defina CONTAGEM_ARTEFATOS para esta pasta e execute streamlit run app.py.")
